In [3]:
# ============================================================
# Explainable Depression Detection — Full Gradio App (Redesigned UI)
# ============================================================

!pip -q install gradio shap

import gradio as gr
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib, json, re
from transformers import AutoTokenizer, AutoModelForSequenceClassification



ARTIFACT_DIR = "depression_detector_artifacts"
MAX_LENGTH = 256

# ---------- Load model + tokenizer ----------
tokenizer = AutoTokenizer.from_pretrained(f"{ARTIFACT_DIR}/model")
model = AutoModelForSequenceClassification.from_pretrained(f"{ARTIFACT_DIR}/model")
model.eval()
id2label = model.config.id2label

# ---------- Load SVM artifacts (for word frequency plot) ----------
vectorizer = joblib.load(f"{ARTIFACT_DIR}/vectorizer.pkl")
feature_svm = joblib.load(f"{ARTIFACT_DIR}/feature_svm.pkl")

# ---------- Load metrics ----------
with open(f"{ARTIFACT_DIR}/metrics.json") as f:
    saved_metrics = json.load(f)
svm_matched_results = saved_metrics["svm_matched_results"]
transformer_test_results = saved_metrics["transformer_test_results"]
cm = np.array(saved_metrics["confusion_matrix"])

# ---------- Load dataset sample ----------
df = pd.read_csv(f"{ARTIFACT_DIR}/df_sample.csv")

# ---------- Preprocessing (MUST match training exactly) ----------
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'[*_~`>#]', ' ', text)
    text = re.sub(r'&amp;|&gt;|&lt;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ---------- SHAP explainer ----------
masker = shap.maskers.Text(tokenizer)

def predict_proba_text(texts):
    cleaned = [clean_text(t) for t in texts]
    encoded = tokenizer(cleaned, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        return torch.softmax(model(**encoded).logits, dim=-1).numpy()

explainer_ui = shap.Explainer(predict_proba_text, masker)

# ============================================================
# TAB 1: Predict + Explain — custom rendered output
# ============================================================

def _confidence_bar_html(probs):
    """Custom two-segment confidence bar instead of default gr.Label."""
    normal_pct = probs[0] * 100
    dep_pct = probs[1] * 100
    return f"""
    <div class="conf-wrap">
        <div class="conf-row">
            <span class="conf-name">✅ Normal</span>
            <span class="conf-pct">{normal_pct:.1f}%</span>
        </div>
        <div class="conf-track">
            <div class="conf-fill conf-fill-normal" style="width:{normal_pct:.1f}%"></div>
        </div>
        <div class="conf-row" style="margin-top:14px;">
            <span class="conf-name">⚠️ Depression-like</span>
            <span class="conf-pct">{dep_pct:.1f}%</span>
        </div>
        <div class="conf-track">
            <div class="conf-fill conf-fill-dep" style="width:{dep_pct:.1f}%"></div>
        </div>
    </div>
    """

def _prediction_badge_html(label_str):
    if label_str == "1":
        return """<div class="badge badge-alert">⚠️ Depression-like</div>"""
    return """<div class="badge badge-ok">✅ Normal</div>"""

def predict_and_explain_ui(text):
    if not text or not text.strip():
        empty_badge = """<div class="badge badge-empty">Enter text to analyze</div>"""
        empty_conf = """<div class="conf-wrap conf-empty">No prediction yet.</div>"""
        empty_html = """<div class="shap-card shap-empty">Your word-level explanation will appear here once you run an analysis.</div>"""
        return empty_badge, empty_conf, empty_html

    cleaned_text = clean_text(text)
    encoded = tokenizer(cleaned_text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)[0].numpy()

    label = str(id2label[int(probs.argmax())])
    badge_html = _prediction_badge_html(label)
    conf_html = _confidence_bar_html(probs)

    shap_values = explainer_ui([cleaned_text])
    raw_shap_html = shap.plots.text(shap_values[:, :, 1], display=False)
    wrapped_shap_html = f"""
    <div class="shap-card">
        <div class="shap-card-title">Word-level contribution to "Depression-like" score</div>
        <div class="shap-card-sub">Darker / redder tokens push the prediction toward depression-like; blue tokens push toward normal.</div>
        <div class="shap-render">{raw_shap_html}</div>
    </div>
    """

    return badge_html, conf_html, wrapped_shap_html

# ============================================================
# TAB 2: Dashboard plots (unchanged)
# ============================================================

def class_distribution_plot():
    counts = df['label'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.pie(counts, labels=['Normal (0)', 'Depression (1)'], autopct='%1.1f%%',
           colors=['#4C72B0', '#DD8452'], startangle=90)
    ax.set_title("Dataset Class Distribution")
    plt.tight_layout()
    return fig

def word_frequency_plot(top_n=15):
    coefs = feature_svm.coef_[0]
    feature_names = np.array(vectorizer.get_feature_names_out())
    top_pos_idx = np.argsort(coefs)[-top_n:][::-1]
    top_neg_idx = np.argsort(coefs)[:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].barh(feature_names[top_pos_idx][::-1], coefs[top_pos_idx][::-1], color='#DD8452')
    axes[0].set_title("Top Words → Depression Class")
    axes[0].set_xlabel("SVM Weight")
    axes[1].barh(feature_names[top_neg_idx][::-1], coefs[top_neg_idx][::-1], color='#4C72B0')
    axes[1].set_title("Top Words → Normal Class")
    axes[1].set_xlabel("SVM Weight")
    plt.tight_layout()
    return fig

def model_comparison_plot():
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    svm_vals = [svm_matched_results[m] for m in metrics]
    tr_vals = [transformer_test_results[m] for m in metrics]
    x = np.arange(len(metrics))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width/2, svm_vals, width, label='TF-IDF + SVM', color='#4C72B0')
    ax.bar(x + width/2, tr_vals, width, label='DistilBERT', color='#DD8452')
    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1)
    ax.set_title("Baseline vs Transformer — Matched Test Set")
    ax.legend()
    plt.tight_layout()
    return fig

def confusion_matrix_plot():
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred: Normal", "Pred: Depression"])
    ax.set_yticklabels(["True: Normal", "True: Depression"])
    ax.set_title("Transformer — Confusion Matrix")
    plt.tight_layout()
    return fig

def text_length_distribution_plot():
    df['text_length'] = df['text_clean'].astype(str).str.len()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df[df['label']==0]['text_length'], bins=50, alpha=0.6, label='Normal', color='#4C72B0', range=(0,2000))
    ax.hist(df[df['label']==1]['text_length'], bins=50, alpha=0.6, label='Depression', color='#DD8452', range=(0,2000))
    ax.set_xlabel("Post Length (characters)")
    ax.set_ylabel("Count")
    ax.set_title("Text Length Distribution by Class")
    ax.legend()
    plt.tight_layout()
    return fig

def build_dashboard_text():
    metrics_md = f"""
### Model Performance (Test Set, N=20,000, matched)

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| TF-IDF + LinearSVC (baseline) | {svm_matched_results['accuracy']:.4f} | {svm_matched_results['precision']:.3f} | {svm_matched_results['recall']:.3f} | {svm_matched_results['f1']:.3f} | {svm_matched_results.get('roc_auc', 0):.3f} |
| **DistilBERT (fine-tuned)** | **{transformer_test_results['accuracy']:.4f}** | **{transformer_test_results['precision']:.3f}** | **{transformer_test_results['recall']:.3f}** | **{transformer_test_results['f1']:.3f}** | **{transformer_test_results.get('roc_auc', 0):.3f}** |
"""
    limitations_md = """
### Known Limitation: Lexical Shortcut on "interest"

SHAP-based error analysis revealed the model under-detects anhedonia symptoms phrased as **"lost interest"**, even though it reliably catches other high-intensity depressive language.

| Test sentence | Confidence (Depression-like) |
|---|---|
| "I lost interest in everything." | 0.55% |
| "I have a lot of interest in everything." | 0.11% |
| "I lost my job and my friends." | 71.34% |
| "I feel hopeless and empty inside." | 99.43% |

**Root cause:** the token *"interest"* carries a strong standalone negative-attribution bias, largely independent of context — a lexical shortcut rather than true compositional understanding.
"""
    return metrics_md, limitations_md

# ============================================================
# Custom CSS
# ============================================================

CUSTOM_CSS = """
:root {
    --bg-app: #0f1220;
    --card-bg: #ffffff;
    --accent-orange: #e0703c;
    --accent-blue: #4C72B0;
    --text-dark: #1a1a2e;
}

.gradio-container {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    max-width: 1180px !important;
    margin: 0 auto !important;
}

/* ---- Hero header ---- */
.hero {
    background: linear-gradient(135deg, #1f2440 0%, #2e2560 100%);
    border-radius: 18px;
    padding: 28px 32px;
    margin-bottom: 18px;
    color: #ffffff;
}
.hero h1 {
    margin: 0 0 6px 0;
    font-size: 26px;
    font-weight: 700;
    color: #ffffff;
}
.hero p {
    margin: 0;
    font-size: 14px;
    color: #cfd3e8;
    line-height: 1.5;
}
.hero .disclaimer-pill {
    display: inline-block;
    margin-top: 12px;
    background: rgba(224, 112, 60, 0.18);
    border: 1px solid rgba(224, 112, 60, 0.5);
    color: #ffd3ba;
    padding: 8px 14px;
    border-radius: 999px;
    font-size: 12.5px;
}

/* ---- Section cards ---- */
.section-card {
    background: var(--card-bg);
    border-radius: 16px;
    padding: 20px 22px;
    box-shadow: 0 2px 10px rgba(20, 20, 50, 0.06);
    border: 1px solid #eceef5;
}
.section-title {
    font-size: 13px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    color: #6b6f8a;
    margin-bottom: 10px;
}

/* ---- Prediction badge ---- */
.badge {
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 22px 16px;
    border-radius: 14px;
    font-size: 22px;
    font-weight: 800;
    letter-spacing: -0.01em;
}
.badge-alert {
    background: linear-gradient(135deg, #fde7de, #ffd9c6);
    color: #a8431a;
    border: 1px solid #f5b691;
}
.badge-ok {
    background: linear-gradient(135deg, #e2f3ea, #cdeedb);
    color: #1c7a4d;
    border: 1px solid #97dcb8;
}
.badge-empty {
    background: #f3f4f8;
    color: #8b8fa8;
    border: 1px dashed #d5d8e6;
    font-size: 15px;
    font-weight: 600;
}

/* ---- Confidence bars ---- */
.conf-wrap { padding: 6px 2px 2px 2px; }
.conf-empty {
    color: #8b8fa8;
    font-size: 13.5px;
    padding: 14px 4px;
    text-align: center;
}
.conf-row {
    display: flex;
    justify-content: space-between;
    font-size: 13.5px;
    font-weight: 600;
    color: var(--text-dark);
    margin-bottom: 6px;
}
.conf-pct { font-variant-numeric: tabular-nums; color: #4b4f6b; }
.conf-track {
    width: 100%;
    height: 10px;
    background: #eef0f7;
    border-radius: 999px;
    overflow: hidden;
}
.conf-fill { height: 100%; border-radius: 999px; transition: width 0.4s ease; }
.conf-fill-normal { background: linear-gradient(90deg, #4C72B0, #6f94cf); }
.conf-fill-dep { background: linear-gradient(90deg, #DD8452, #e6a679); }

/* ---- SHAP card ---- */
.shap-card {
    background: #fafbff;
    border: 1px solid #e7e9f4;
    border-radius: 14px;
    padding: 18px 20px;
}
.shap-card-title { font-weight: 700; font-size: 14px; color: var(--text-dark); margin-bottom: 4px; }
.shap-card-sub { font-size: 12.5px; color: #7d8098; margin-bottom: 14px; }
.shap-render { overflow-x: auto; }
.shap-empty {
    color: #9498ac;
    font-size: 13.5px;
    text-align: center;
    padding: 30px 10px;
}

/* ---- Analyze button ---- */
#analyze-btn {
    background: linear-gradient(135deg, #e0703c, #d85a2c) !important;
    border: none !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    border-radius: 12px !important;
    box-shadow: 0 4px 14px rgba(224, 112, 60, 0.35);
}

/* ---- Input textbox ---- */
#input-text textarea {
    border-radius: 12px !important;
    font-size: 14.5px !important;
}

/* ---- Examples ---- */
.example-title {
    font-size: 12.5px;
    font-weight: 700;
    color: #6b6f8a;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    margin: 4px 0 6px 2px;
}
"""

# ============================================================
# Build the app
# ============================================================

with gr.Blocks(title="Explainable Depression Detection", css=CUSTOM_CSS, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class="hero">
        <h1>🧠 Explainable Depression Detection</h1>
        <p>A research demo comparing a TF-IDF/SVM baseline against a fine-tuned DistilBERT classifier,
        with SHAP-based word-level explanations for every prediction.</p>
    </div>
    """)

    with gr.Tabs():
        # ---------------- Predict Tab ----------------
        with gr.Tab("🔍 Predict"):
            with gr.Row():
                # Left column — input
                with gr.Column(scale=5):
                    with gr.Group(elem_classes="section-card"):
                        gr.Markdown("**Analyze a post**", elem_classes="section-title")
                        text_input = gr.Textbox(
                            lines=7,
                            placeholder="Type or paste a social media–style post here...",
                            label=None,
                            show_label=False,
                            elem_id="input-text",
                        )
                        predict_btn = gr.Button("Analyze", variant="primary", elem_id="analyze-btn")

                        gr.Markdown("Try an example", elem_classes="example-title")
                        gr.Examples(
                            examples=[
                                ["I feel hopeless and empty inside."],
                                ["Just got back from an amazing hike with friends, feeling great!"],
                                ["I've been feeling really low lately and lost interest in everything."],
                            ],
                            inputs=text_input,
                        )

                # Right column — results
                with gr.Column(scale=6):
                    with gr.Group(elem_classes="section-card"):
                        gr.Markdown("**Prediction**", elem_classes="section-title")
                        pred_label = gr.HTML(
                            value="""<div class="badge badge-empty">Enter text to analyze</div>"""
                        )
                        gr.Markdown("**Confidence**", elem_classes="section-title")
                        pred_confidence = gr.HTML(
                            value="""<div class="conf-wrap conf-empty">No prediction yet.</div>"""
                        )

            with gr.Row():
                with gr.Column():
                    pred_explanation = gr.HTML(
                        value="""<div class="shap-card shap-empty">Your word-level explanation will appear here once you run an analysis.</div>"""
                    )

            predict_btn.click(
                fn=predict_and_explain_ui,
                inputs=text_input,
                outputs=[pred_label, pred_confidence, pred_explanation],
            )

        # ---------------- Analysis Dashboard Tab (unchanged) ----------------
        with gr.Tab("📊 Analysis Dashboard"):
            gr.Markdown("Model evaluation results and dataset analysis.")
            metrics_text, limitations_text = build_dashboard_text()
            gr.Markdown(metrics_text)

            with gr.Row():
                gr.Plot(value=model_comparison_plot())
                gr.Plot(value=confusion_matrix_plot())

            with gr.Row():
                gr.Plot(value=class_distribution_plot())
                gr.Plot(value=text_length_distribution_plot())

            gr.Markdown("### Word Frequency Analysis (Global Feature Importance)")
            gr.Plot(value=word_frequency_plot())

            gr.Markdown(limitations_text)

demo.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

c:\Users\USER-PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\USER-PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\USER-PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickl

* Running on local URL:  http://127.0.0.1:7861


ConnectTimeout: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond